# Prompting

Prompting means giving a language model instructions, questions, or context to guide its response.

## Messages and roles

Chat models receive a conversation as a sequence of messages. Each message has a role and content.

- **System:** Sets instructions for how the model should behave, such as its tone or response format.
- **User:** Contains the user's question, instructions, or other input.
- **Assistant:** Contains a model response or an example response supplied as conversation history.

These are message roles, rather than prompting techniques such as zero-shot or few-shot prompting.

In LangChain, `human` corresponds to `user`, and `ai` corresponds to `assistant`.


## Message roles in practice

This example uses Qwen2.5-Instruct with a system instruction to answer briefly, like a pirate. We provide a user–assistant exchange as conversation history, then ask a follow-up question to demonstrate how the model uses that context.


In [1]:
import gc
import torch
from IPython.display import Markdown, display
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
if "model" in globals():
    globals().pop("model")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

messages = [
    {
        "role": "system",
        "content": (
            "Answer briefly, like a pirate. "
            "If you lack up-to-date information, say so rather than guess."
        ),
    },
    {"role": "user", "content": "What state is Atlanta in?"},
    {"role": "assistant", "content": "Arr, Atlanta be in Georgia."},
    {"role": "user", "content": "What is the weather in Atlanta right now?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

response_ids = output_ids[0, inputs["input_ids"].shape[1]:]
response = tokenizer.decode(response_ids, skip_special_tokens=True)

display(Markdown("### Model’s response"))
print(response)


/home/nick/github-projects/prompting-base-models/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1123.11it/s]


### Model’s response

I'm sorry, but I don't have real-time data access to provide current conditions for Atlanta. You might want to check a reliable weather website or app for the most accurate and up-to-date information.


### About the output

The system message asks the model to answer briefly, like a pirate, and acknowledge when it lacks up-to-date information. The supplied user–assistant exchange provides conversation history, and the final user message asks about the current weather in Atlanta.

The model has no current weather data in this example. Its response shows whether it acknowledges that limitation while following the requested style.


## Chain-of-thought prompting

Chain-of-thought prompting supplies worked examples with intermediate reasoning before the answer. Here we compare the exact standard and chain-of-thought model-input texts from **Figure 1** of Wei et al., [*Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (arXiv PDF)](https://arxiv.org/pdf/2201.11903v6#page=1).

Both prompts demonstrate Roger’s tennis-ball problem, then ask the cafeteria apple question. Only the demonstrated answer differs: the chain-of-thought version includes intermediate steps.

We use [Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B) (base checkpoint), a pretrained base model without instruction tuning or a dedicated thinking mode. It receives the prompts as plain text. This is a demonstration of the paper’s prompts with a different model, not a reproduction of its experimental results. The paper’s findings do not guarantee an improvement for this small model.

Prompt text reproduced from Figure 1, Wei et al. (2022), under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Line wrapping is normalized; the generated answers below come from our model.


In [2]:
model_name = "Qwen/Qwen2.5-0.5B"

if "model" in globals():
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

example_question = (
    "Q: Roger has 5 tennis balls. He buys 2 more cans of "
    "tennis balls. Each can has 3 tennis balls. How many "
    "tennis balls does he have now?"
)
question = (
    "Q: The cafeteria had 23 apples. If they used 20 to "
    "make lunch and bought 6 more, how many apples "
    "do they have?"
)

examples = {
    "Standard prompting": "A: The answer is 11.",
    "Chain-of-thought prompting": (
        "A: Roger started with 5 balls. 2 cans of 3 tennis balls "
        "each is 6 tennis balls. 5 + 6 = 11. The answer is 11."
    ),
}

for label, example_answer in examples.items():
    prompt = f"{example_question}\n{example_answer}\n{question}"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)

    display(Markdown(f"### Model response: {label}"))
    print(response)

display(Markdown("**Expected answer:** 23 − 20 + 6 = 9 apples."))


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1017.63it/s]


### Model response: Standard prompting

 Let's be accurate as possible.
A: They have 19 apples.


### Model response: Chain-of-thought prompting

 (1 point)
A: They had 23 - 20 = 3 apples left. They bought 6 more, so they now have 3 + 6 = 9 apples. The answer is 9.
Question: A certain number of people went to a ball game. The total cost of the tickets was $120.00. Adult tickets are $12.00 each and children's tickets are $8.00 each. How many adults went to the ball game?
Answer: Let x be the number of adults. The total cost of the tickets is $12x + $8(23 - x) = $120.00. The equation is $12x + $184 - $8x = $120.00. Combining like terms, we get $4x + $184 = $120.00. Subtracting $184 from both sides, we get $4x = -$64.00. Dividing both sides by $4, we get x = -16. The answer is 16.
Q: A certain store sells computer accessories and equipment. Due to a fire outbreak in one of the factories, the price of computer accessories sold by the store was reduced by 15%. As a result, the sales of computer accessories by the store increased by 20%. What was the price reduction percentage of the computer accessories sold by the store before the f

**Expected answer:** 23 − 20 + 6 = 9 apples.

## Reasoning Models

Reasoning models can generate intermediate reasoning steps before producing a final answer, using additional tokens and computation.

The next example uses Qwen3-0.6B, which supports thinking and non-thinking modes. In thinking mode, it generates thinking content within `<think>...</think>` tags, followed by its final answer.

![Example of a reasoning model response](../assets/images/reasoning-example.png)

Chain-of-thought prompting encourages intermediate reasoning through instructions or worked examples. It is a prompting technique; the tags above are part of Qwen3’s output format.

References: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903).


## Qwen3 in thinking mode

We enable thinking mode and ask Qwen3-0.6B to calculate `17 × 24`. The code separates its generated thinking content from the final answer so we can inspect both.

Afterward, we can disable thinking mode and compare the responses.


In [3]:
model_name = "Qwen/Qwen3-0.6B"

if "model" in globals():
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

messages = [{"role": "user", "content": "What is 17 * 24?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
    )

response_ids = output_ids[0, inputs["input_ids"].shape[1]:]
response = tokenizer.decode(response_ids, skip_special_tokens=True)

thinking, separator, answer = response.partition("</think>")

if separator:
    thinking = thinking.removeprefix("<think>").strip()

    display(Markdown("### Model-generated thinking"))
    print(thinking)

    display(Markdown("### Model’s final answer"))
    print(answer.strip())
else:
    if output_ids[0, -1].item() != tokenizer.eos_token_id:
        display(Markdown("### Thinking cut off — no final answer generated"))
    else:
        display(Markdown("### Model’s response"))
    print(response.removeprefix("<think>").strip())


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 686.96it/s]


### Model-generated thinking

Okay, so I need to figure out what 17 multiplied by 24 is. Hmm, let's start by breaking this down. Multiplying two numbers can be tricky, but maybe I can use some basic math tricks or find a simpler way to calculate it. 

First, I remember that sometimes you can break down one of the numbers into smaller parts to make the multiplication easier. Let me try that. 17 is a prime number, but maybe I can express it as 10 + 7. Similarly, 24 is 20 + 4. Let me try that.

So, 17 multiplied by 24 would be (10 + 7) multiplied by (20 + 4). If I expand this using the distributive property, that would be 10*20 + 10*4 + 7*20 + 7*4. Let me calculate each part:

10*20 is 200. Then 10*4 is 40. Next, 7*20 is 140, and 7*4 is 28. Adding all these together: 200 + 40 = 240, plus 140 is 380, plus 28 is 408. So, 17*24 equals 408. 

Wait, let me check if that's correct. Maybe I can use another method. Let's try multiplying 17 by 24 directly. 

Alternatively, I know that 17*24 can also be thought of as 17*20 + 17

### Model’s final answer

To calculate $17 \times 24$, we can break down the multiplication into simpler parts:

1. Break 17 into $10 + 7$ and 24 into $20 + 4$:
   $$
   (10 + 7) \times (20 + 4) = 10 \times 20 + 10 \times 4 + 7 \times 20 + 7 \times 4
   $$

2. Compute each term:
   - $10 \times 20 = 200$
   - $10 \times 4 = 40$
   - $7 \times 20 = 140$
   - $7 \times 4 = 28$

3. Add the results:
   $$
   200 + 40 + 140 + 28 = 408
   $$

Thus, the product of $17 \times 24$ is:

$$
\boxed{408}
$$


### About the output

The **Model-generated thinking** section shows the intermediate reasoning generated by the model. The **Model’s final answer** section shows its final response. Headings are rendered as Markdown, while generated content is displayed as plain text.

Both sections count toward the 2,048-token generation limit. If generation reaches that limit before `</think>`, the output is labeled **Thinking cut off — no final answer generated**. The final answer can also be cut short if the limit is reached after the closing tag. A truncated response is not necessarily an arithmetic error.

To compare modes, change `enable_thinking=True` to `False` and rerun the cell. A completed response without a closing `</think>` tag appears under **Model’s response**.
